In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
#ddavila: add a brief description of what the data is
df_cleaned = pd.read_csv('df_cleaned.csv')

In [ ]:
#Units transferring
#ddavila: this column is not used anywhere
df_cleaned['Gb/s'] = df_cleaned['throughput'] * 8 / 1000000000

#ddavila: All filters together, bring the ASO filter here
df_cleaned = df_cleaned[df_cleaned['transferred_volume'] != 0]
df_cleaned = df_cleaned[df_cleaned['throughput'] > 0]

#ddavila: can be simplified as :df_cleaned['t_final_transfer_state']="Ok"
df_cleaned = df_cleaned[
    df_cleaned['t_final_transfer_state'].astype(str).str.strip().str.upper() == 'OK'
]

In [ ]:
#ddavila: many unecessary DataFrame copies df_cleaned => unl => fts
unl = df_cleaned
#Check if there is any duplicate records
#ddavila: use drop_duplicates using inPlace=True to avo
mask_dup = unl.duplicated(subset=['transferred_volume', 'timestamp_tr_st','dst_url'], keep=False)
df_dup = unl[mask_dup]

unl = df_cleaned
fts = unl.copy()
# Filter out src is UNKNOWN results and duplicate if any
fts['start_dt'] = pd.to_datetime(unl['timestamp_tr_st'], unit='ms', utc=True)
fts['end_dt'] = pd.to_datetime(unl['timestamp_tr_comp'], unit='ms', utc=True)

#ddavila: column 'duration' never used again
fts['duration'] = (fts['end_dt'] - fts['start_dt']).dt.total_seconds()
fts = fts.drop(df_dup.index)
fts = fts[fts['src_experiment_site'] != 'UNKNOWN']

In [ ]:
rtt_intervals = [
    (10, 20),
    (20, 25),
    (25, 30),
    (30, 35),
    (35, 40),
    (40, 45),
    (45, 50),
    (50, 55),
    (55, 60),
    (60, 150),
]
site_rtt_dict = {
    "T2_US_Caltech": 37.4,
    "T2_US_Florida": 51.6,
    "T2_US_MIT": 34.5,
    "T2_US_Purdue": 20.7,
    "T2_US_Vanderbilt": 28.9,
    "T2_US_Wisconsin": 33.1,
    "T2_US_UCSD": 39.2,
    "T1_US_FNAL": 17.1,
    "T0_CH_CERN": 127.0,
}

In [ ]:
df_cleaned["rtt"] = df_cleaned["src_experiment_site"].map(site_rtt_dict)
#ddavila: keep filters together on top. Also more DataFrame copies, do "inPlace" operations as much as possible to avoid copying  
rtt = df_cleaned.dropna(subset=["rtt"]).copy()
rtt_clean = rtt[rtt["activity"] != "ASO"].copy()

In [ ]:
# ddavila: keep long functions on separated files or at least move above (after imports)
def classify_rtt_bucket(rtt, intervals):
    for low, high in intervals:
        if low <= rtt < high:
            return f"active_transfers_rtt_{low}_{high}"
    return "active_transfers_rtt_other"
def compute_throughput(df, time_points):
    records = []
    for i in range(len(time_points) - 1):
        # window start
        t_start = time_points[i]
        # window ends = start time + 1 min
        t_end = time_points[i + 1]
        # Determine active transfer count: the transfer started at or before t_start, and ends after t_start
        # at the exact time, concurrent transfers
        time_mask = (df['start_dt'] < t_end) & (df['end_dt'] > t_start)
        active = df[time_mask]
        # If no transfers are active in this window, skip it
        if len(active) == 0:
            records.append(0.0)
            continue
        # Compute the throughput within this window
        overlap_start = active['start_dt'].clip(lower=t_start)
        overlap_end = active['end_dt'].clip(upper=t_end)
        # How long this transfer is actually active within the window
        overlap_sec = (overlap_end - overlap_start).dt.total_seconds()
        total_sec = (active['end_dt'] - active['start_dt']).dt.total_seconds()
        bytes_in_window = active['transferred_volume'] * (overlap_sec / total_sec)
        #Assuming a uniform transfer rate, allocate the total volume to this window (and unit calculation)
        records.append(bytes_in_window.sum() / 60 )  # bytes/s
    return records

def compute_active(df, time_points):
    counts = []
    for i in range(len(time_points) - 1):
        t_start = time_points[i]
        t_end = time_points[i + 1]
        # count every transfer that exists at any moment within [t_start, t_end)
        time_mask = (df['start_dt'] < t_end) & (df['end_dt'] > t_start)
        counts.append(int(time_mask.sum()))
    return counts

In [ ]:
#ddavila "time_col" var used only once. The column is never used
time_col = "timestamp_tr_st"
rtt_clean[time_col] = pd.to_datetime(rtt_clean[time_col], unit="ms")
rtt_clean["timestamp_tr_comp"] = pd.to_datetime(rtt_clean["timestamp_tr_comp"], unit="ms")
rtt_clean["start_dt"] = rtt_clean["timestamp_tr_st"]
rtt_clean["end_dt"] = rtt_clean["timestamp_tr_comp"]

# ddavila: Classifies the tx per rtt bucket
rtt_clean["rtt_bucket"] = rtt_clean["rtt"].apply(
    lambda x: classify_rtt_bucket(x, rtt_intervals)
)

#ddavila: the next 2 blocks creates the set of time windows to be used to calculate
# active transfers (1hr) and throughput (1m)

#Get time axis, make the start time floored to the minute, and end time celled to minute.
time_points = pd.date_range(
    start=rtt_clean['start_dt'].min().floor('min'),
    end=rtt_clean['end_dt'].max().ceil('min'),
    freq='1min',
)

# hourly time axis for active transfer count
hour_points = pd.date_range(
    start=rtt_clean['start_dt'].min().floor('h'),
    end=rtt_clean['end_dt'].max().ceil('h'),
    freq='1h',
)

In [ ]:
#ddavila: Add comments: calls the functions that calculate active transfers and throughput per bucket 
tp_cols, active_cols = {}, {}
for bucket, g in rtt_clean.groupby("rtt_bucket", observed=True):
    tp_cols[bucket] = compute_throughput(g, time_points)
    active_cols[bucket] = compute_active(g, hour_points)

In [ ]:
# ddavila: Add comments: creates a new DataFrame based on the dictionaries created above
df_processed = pd.concat(
    {
        "active_transfers": pd.DataFrame(active_cols, index=hour_points[:-1]),
        "throughput": pd.DataFrame(tp_cols, index=time_points[:-1]).resample("60Min").mean(),
    },
    axis=1,
).fillna(0)

In [ ]:
# ddavila: Add comments: The next few cells just change the name and order of the columns (formating)
df_processed.columns = [
    f"{metric}_{bucket.replace('active_transfers_', '')}"
    for metric, bucket in df_processed.columns
]

In [ ]:
buckets = sorted(
    {c.replace("active_transfers_", "").replace("throughput_", "")
     for c in df_processed.columns},
    key=lambda b: int(b.split("_")[1])
)

In [ ]:
ordered_cols = []
for b in buckets:
    ordered_cols += [f"active_transfers_{b}", f"throughput_{b}"]

In [ ]:
df_processed = df_processed[ordered_cols]
df_processed[[c for c in ordered_cols if c.startswith("active_transfers_")]] = \
    df_processed[[c for c in ordered_cols if c.startswith("active_transfers_")]].astype(int)

In [ ]:
# ddavila: Add comments: This adds the total throuhgput column
# ddavila: not sure why you need the following 'if' statement
if "transferred_volume" in rtt_clean.columns:
    df_throughput = pd.Series(
        compute_throughput(rtt_clean, time_points), index=time_points[:-1]
    ).resample("60Min").mean()
    df_model_data = df_processed.join(df_throughput.rename("total_throughput"))
else:
    df_model_data = df_processed

In [ ]:
throughput_cols = [c for c in df_model_data.columns if c.startswith("throughput_rtt")]

fig, ax = plt.subplots(figsize=(14, 6))

for col in throughput_cols:
    ax.plot(df_model_data.index, df_model_data[col],
            label=col.replace("throughput_", ""), linewidth=1.2, alpha=0.8)

ax.plot(df_model_data.index, df_model_data["total_throughput"],
        label="total", color="black", linewidth=2.2, linestyle="--")

ax.set_xlabel("Time")

# ddavila: always mention units in plots!
ax.set_ylabel("Throughput")
ax.legend(title="RTT Bucket", loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
feature_cols = [c for c in df_model_data.columns if c.startswith("active_transfers_")]

X = df_model_data[feature_cols]
y = df_model_data["total_throughput"]

mask = y.notna() & (y > 0)
X, y = X[mask], y[mask]

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error
kf = KFold(n_splits=8, shuffle=True, random_state=42)
results, weights_per_fold = [], []

for i, (tr_i, te_i) in enumerate(kf.split(df_model_data)):
    train, test = df_model_data.iloc[tr_i], df_model_data.iloc[te_i]

    w = pd.Series({b: train[f"throughput_{b}"].sum() / train[f"active_transfers_{b}"].sum()
                for b in buckets})
    weights_per_fold.append(w.rename(f"fold_{i}"))

    y_hat = sum(test[f"active_transfers_{b}"] * w[b] for b in buckets)
    y_t = test["total_throughput"]
    results.append({"fold": i, "R2": r2_score(y_t, y_hat),
                    "WAPE": (y_hat - y_t).abs().sum() / y_t.sum()})

print(pd.DataFrame(results).set_index("fold").round(3))

In [ ]:
wdf_clean = pd.concat(weights_per_fold, axis=1)

In [ ]:
#Proof of weight is almost constant
fig, ax = plt.subplots(figsize=(12, 6))

for b in wdf_clean.index:
    ax.plot(wdf_clean.columns, wdf_clean.loc[b] / 1e6, marker="o", label=b, linewidth=1.5)
    ax.axhline(wdf_clean.loc[b].mean() / 1e6, color="gray", linewidth=0.5,
               linestyle=":", alpha=0.5)

ax.set_ylim(0, None)
ax.set_xlabel("CV fold")
ax.set_ylabel("weight (×10⁶, throughput per transfer)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
#ddavila: why you need the next line?
print([c for c in rtt.columns if "activity" in c.lower()])
print(rtt["activity"].value_counts())

In [ ]:
by_activity = rtt.groupby("activity").agg(
    n_transfers=("throughput", "size"),
    total_throughput=("throughput", "sum"),

)

by_activity["n_share_%"] = (by_activity["n_transfers"] / by_activity["n_transfers"].sum() * 100).round(2)
by_activity["tp_share_%"] = (by_activity["total_throughput"] / by_activity["total_throughput"].sum() * 100).round(2)

by_activity.sort_values("n_transfers", ascending=False)